In [ ]:
import pandas as pd

# 1. Carregar a base de leitos
caminho = r'C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw\cnes_leitos_sp.csv'
df_leitos = pd.read_csv(caminho, sep=';', encoding='latin1')

# 2. Separar o código do IBGE e o nome do município
df_leitos['Cod_IBGE'] = df_leitos['Município'].astype(str).str.split(' ').str[0]
df_leitos['Nome_Municipio'] = df_leitos['Município'].astype(str).str.split(' ', n=1).str[1]
df_leitos['Quantidade_Leitos'] = pd.to_numeric(df_leitos['Quantidade_existente'], errors='coerce').fillna(0)

# 3. Filtrar APENAS os municípios de São Paulo (código IBGE começa com '35')
df_leitos_sp = df_leitos[df_leitos['Cod_IBGE'].str.startswith('35')].copy()
df_leitos_sp = df_leitos_sp[['Cod_IBGE', 'Nome_Municipio', 'Quantidade_Leitos']]

print(f"Total de municípios de SP encontrados: {len(df_leitos_sp)}")
print("Soma total de leitos em SP:", df_leitos_sp['Quantidade_Leitos'].sum())

# 4. Salvar o arquivo limpo
df_leitos_sp.to_csv('cnes_leitos_sp_tratado.csv', index=False, encoding='utf-8-sig')
print("\nBase tratada salva com sucesso como 'cnes_leitos_sp_tratado.csv'!")

Total de municípios de SP encontrados: 355
Soma total de leitos em SP: 96589

Base tratada salva com sucesso como 'cnes_leitos_sp_tratado.csv'!


In [ ]:
import pandas as pd
import glob
import os

# 1. O caminho 
caminho_dados = r'C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw'
padrao_busca = os.path.join(caminho_dados, 'sih_*.csv')
arquivos_sih = sorted(glob.glob(padrao_busca))

print(f"Total de arquivos SIH encontrados para processamento: {len(arquivos_sih)}")

lista_dfs = []

# 2. Processamento direto 
for arquivo in arquivos_sih:
    try:
        # O Pandas já lê as 8 colunas separadas perfeitamente por causa do sep=';'
        df_sih = pd.read_csv(arquivo, sep=';', encoding='latin1')
        
        # O DATASUS traz o código e o nome juntos na coluna "Município" ("350010 ADAMANTINA")
        # Vamos separar isso em duas colunas novas
        df_sih['Cod_IBGE'] = df_sih['Município'].astype(str).str.split(' ', n=1).str[0]
        df_sih['Nome_Municipio'] = df_sih['Município'].astype(str).str.split(' ', n=1).str[1]
        
        lista_dfs.append(df_sih)
    except Exception as e:
        print(f"Erro ao processar o arquivo {arquivo}: {e}")

# 3. Consolidação e limpeza final
if len(lista_dfs) > 0:
    df_sih_geral = pd.concat(lista_dfs, ignore_index=True)
    
    # Filtrar apenas o Estado de São Paulo
    df_sih_sp = df_sih_geral[df_sih_geral['Cod_IBGE'].str.startswith('35')].copy()
    
    # Converter números (trocando vírgula por ponto e tratanto os '-' vazios do DATASUS como zero)
    cols_numericas = ['AIH_aprovadas', 'Internações', 'Dias_permanência', 'Óbitos']
    for col in cols_numericas:
        df_sih_sp[col] = pd.to_numeric(
            df_sih_sp[col].astype(str).str.replace(',', '.').str.replace('-', '0').str.strip(), 
            errors='coerce'
        ).fillna(0)
        
    # Renomeando colunas para facilitar lá no Power BI
    df_sih_sp = df_sih_sp.rename(columns={'Internações': 'Internacoes', 'Dias_permanência': 'Dias_permanencia'})
    
    # Mantendo apenas o que importa
    colunas_finais = ['Cod_IBGE', 'Nome_Municipio', 'AIH_aprovadas', 'Internacoes', 'Dias_permanencia', 'Óbitos', 'Período']
    df_sih_sp = df_sih_sp[colunas_finais]
        
    print(f"\nTotal de registros de internações de SP consolidados (36 meses): {len(df_sih_sp)}")
    display(df_sih_sp.head())
    
    # 4. Salva o CSV final
    caminho_saida = os.path.join(caminho_dados, 'sih_sp_consolidado.csv')
    df_sih_sp.to_csv(caminho_saida, index=False, encoding='utf-8-sig')
    print(f"\nBase de internações consolidada salva com sucesso em: {caminho_saida}")
else:
    print("Nenhum arquivo processado. Verifique o caminho.")

Total de arquivos SIH encontrados para processamento: 36

Total de registros de internações de SP consolidados (36 meses): 11656


,Cod_IBGE,Nome_Municipio,AIH_aprovadas,Internacoes,Dias_permanencia,Óbitos,Período
0,350010,ADAMANTINA,479,397,4627,29,01/01/2023
1,350030,AGUAI,7,7,11,0,01/01/2023
2,350050,AGUAS DE LINDOIA,107,107,364,9,01/01/2023
3,350070,AGUDOS,73,73,135,1,01/01/2023
4,350100,ALTINOPOLIS,93,93,184,2,01/01/2023



Base de internações consolidada salva com sucesso em: C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw\sih_sp_consolidado.csv


In [ ]:
import pandas as pd
import os

caminho_dados = r'C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw'


# 1. Ler a base do SIH da pasta do OneDrive
df_sih = pd.read_csv(os.path.join(caminho_dados, 'sih_sp_consolidado.csv'))
# Lê a base de leitos da pasta atual do Jupyter (onde salvamos no Passo 1)
df_leitos = pd.read_csv('cnes_leitos_sp_tratado.csv')

# Garantir que o Código IBGE seja texto/string sem espaços para o cruzamento perfeito
df_sih['Cod_IBGE'] = df_sih['Cod_IBGE'].astype(str).str.strip()
df_leitos['Cod_IBGE'] = df_leitos['Cod_IBGE'].astype(str).str.strip()

# 2. Fazer o cruzamento (Merge) das internações com a capacidade de leitos
# Usamos left join para manter todos os históricos de internação intactos
df_final = pd.merge(
    df_sih,
    df_leitos[['Cod_IBGE', 'Quantidade_Leitos']],
    on='Cod_IBGE',
    how='left'
)

# Se algum município não tiver leitos registrados no CNES, preenchemos com 0 para não quebrar a conta
df_final['Quantidade_Leitos'] = df_final['Quantidade_Leitos'].fillna(0)

# 3. Calcular o IPA (Índice de Pressão Assistencial)
# Lógica: (Internações / Leitos) * 100. Adicionamos proteção contra divisão por zero.
def calcular_ipa(row):
    if row['Quantidade_Leitos'] > 0:
        return (row['Internacoes'] / row['Quantidade_Leitos']) * 100
    return 0

df_final['IPA'] = df_final.apply(calcular_ipa, axis=1)

# Definir Status de Risco (Regras de Negócio do seu MVP)
def classifica_risco(ipa):
    if ipa > 100:
        return 'Crítico'
    elif ipa >= 75:
        return 'Alerta'
    else:
        return 'Estável'

df_final['Status_Risco'] = df_final['IPA'].apply(classifica_risco)

# Converter a coluna Período para data (ajuda muito o Power BI a reconhecer meses/anos)
df_final['Período'] = pd.to_datetime(df_final['Período'], format='%d/%m/%Y', errors='coerce')

print(f"Base Analítica Final gerada com sucesso! Total de linhas: {len(df_final)}")
display(df_final.head())

# 4. Salvar o arquivo final consolidado que vai pro Power BI
caminho_final = os.path.join(caminho_dados, 'healthops_base_analitica_mvp.csv')
df_final.to_csv(caminho_final, index=False, encoding='utf-8-sig', sep=';')

print(f"\nArquivo final salvo em: {caminho_final}")

Base Analítica Final gerada com sucesso! Total de linhas: 11656


,Cod_IBGE,Nome_Municipio,AIH_aprovadas,Internacoes,Dias_permanencia,Óbitos,Período,Quantidade_Leitos,IPA,Status_Risco
0,350010,ADAMANTINA,479,397,4627,29,2023-01-01,194,204.639175,Crítico
1,350030,AGUAI,7,7,11,0,2023-01-01,24,29.166667,Estável
2,350050,AGUAS DE LINDOIA,107,107,364,9,2023-01-01,42,254.761905,Crítico
3,350070,AGUDOS,73,73,135,1,2023-01-01,36,202.777778,Crítico
4,350100,ALTINOPOLIS,93,93,184,2,2023-01-01,21,442.857143,Crítico



Arquivo final salvo em: C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw\healthops_base_analitica_mvp.csv


In [ ]:
import pandas as pd
import os
from pandas.tseries.offsets import DateOffset

# 1. Caminho e carregamento da base 
caminho_dados = r'C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw'
arquivo_base = os.path.join(caminho_dados, 'healthops_base_analitica_mvp.csv')

df = pd.read_csv(arquivo_base, sep=';')
df['Período'] = pd.to_datetime(df['Período'])

# Criar a coluna de classificação para o Power BI
df['Tipo_Dado'] = 'Realizado'

print("Calculando projeções preditivas para os próximos 6 meses...")

# 2. Descobrir o último mês da nossa base (para começar a projetar a partir dali)
ultimo_mes = df['Período'].max()

# 3. Calcular o comportamento recente de cada município (Média Móvel e Últimos Leitos)
resumo_municipios = df.groupby(['Cod_IBGE', 'Nome_Municipio']).agg(
    Media_Internacoes=('Internacoes', lambda x: x.tail(6).mean()), # Média dos últimos 6 meses
    Leitos_Atuais=('Quantidade_Leitos', 'last') # Leitos atuais
).reset_index()

# 4. Gerar os dados do futuro (Próximos 6 meses)
projecoes = []

for i in range(1, 7):
    mes_futuro = ultimo_mes + DateOffset(months=i)
    
    for _, row in resumo_municipios.iterrows():
        # Lógica Preditiva do MVP: Média histórica + tendência de aumento de 2% ao mês
        fator_crescimento = 1 + (0.02 * i) 
        internacoes_projetadas = int(row['Media_Internacoes'] * fator_crescimento)
        
        # Ignorar municípios que não tiveram histórico de internação
        if internacoes_projetadas > 0:
            projecoes.append({
                'Cod_IBGE': row['Cod_IBGE'],
                'Nome_Municipio': row['Nome_Municipio'],
                'AIH_aprovadas': internacoes_projetadas,
                'Internacoes': internacoes_projetadas,
                'Dias_permanencia': internacoes_projetadas * 4, # Estimativa: 4 dias por paciente
                'Óbitos': int(internacoes_projetadas * 0.05), # Estimativa: taxa de 5%
                'Período': mes_futuro,
                'Quantidade_Leitos': row['Leitos_Atuais'],
                'Tipo_Dado': 'Projetado'
            })

df_projecoes = pd.DataFrame(projecoes)

# 5. Calcular o IPA e o Status de Risco PREDITIVOS
df_projecoes['IPA'] = df_projecoes.apply(
    lambda r: (r['Internacoes'] / r['Quantidade_Leitos'] * 100) if r['Quantidade_Leitos'] > 0 else 0, axis=1
)

def classifica_risco(ipa):
    if ipa > 100: return 'Crítico'
    elif ipa >= 75: return 'Alerta'
    else: return 'Estável'

df_projecoes['Status_Risco'] = df_projecoes['IPA'].apply(classifica_risco)

# 6. Unir o Passado (Realizado) com o Futuro (Projetado)
df_final_completo = pd.concat([df, df_projecoes], ignore_index=True)

# 7. Salvar a super base mestre
caminho_projecao = os.path.join(caminho_dados, 'healthops_base_completa_com_projecoes.csv')
df_final_completo.to_csv(caminho_projecao, index=False, encoding='utf-8-sig', sep=';')

print(f"\nSucesso! Foram geradas {len(df_projecoes)} linhas de projeções futuras.")
print(f"Base Mestre com IA Preditiva salva em: {caminho_projecao}")

Calculando projeções preditivas para os próximos 6 meses...

Sucesso! Foram geradas 1974 linhas de projeções futuras.
Base Mestre com IA Preditiva salva em: C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw\healthops_base_completa_com_projecoes.csv


In [ ]:
import pandas as pd
import os

caminho_dados = r'C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw'

# 1. Carregar a Base Mestre (com histórico e projeções) 
caminho_projecao = os.path.join(caminho_dados, 'healthops_base_completa_com_projecoes.csv')
df_mestre = pd.read_csv(caminho_projecao, sep=';')

# 2. Carregar e Limpar os dados de População do IBGE
# A base do IBGE tem um título na primeira linha, então pulamos ela (skiprows=1)
caminho_ibge = os.path.join(caminho_dados, 'ibge_populacao_sp.csv')
df_ibge = pd.read_csv(caminho_ibge, skiprows=1, encoding='utf-8', on_bad_lines='skip')

# Criando um DataFrame limpo apenas com o que importa do IBGE
df_pop_limpa = pd.DataFrame()

# O IBGE usa código de 7 dígitos (ex: 3500105), mas o DATASUS usa 6 (ex: 350010)
# Vamos cortar o último dígito para que o cruzamento (PROCV) funcione perfeitamente!
df_pop_limpa['Cod_IBGE'] = df_ibge.iloc[:, 1].astype(str).str[:6] 

# Pegando a "População estimada [2025]" (que está na 8ª coluna, índice 7 do Pandas)
df_pop_limpa['Populacao'] = pd.to_numeric(df_ibge.iloc[:, 7], errors='coerce').fillna(0)

# 3. Cruzar a População com a Base Mestre
df_mestre['Cod_IBGE'] = df_mestre['Cod_IBGE'].astype(str).str.strip()
df_final_enriquecido = pd.merge(
    df_mestre,
    df_pop_limpa,
    on='Cod_IBGE',
    how='left'
)

# 4. Criar a Métrica de Ouro da Saúde Pública (Internações por 100 mil habitantes)
# Isso nivela municípios grandes e pequenos na mesma régua!
def taxa_por_100k(row):
    if row['Populacao'] > 0:
        return (row['Internacoes'] / row['Populacao']) * 100000
    return 0

df_final_enriquecido['Taxa_Internacao_100k'] = df_final_enriquecido.apply(taxa_por_100k, axis=1)

# 5. Salvar a base hiper-enriquecida
caminho_saida_final = os.path.join(caminho_dados, 'healthops_base_completa_com_projecoes.csv')
df_final_enriquecido.to_csv(caminho_saida_final, index=False, encoding='utf-8-sig', sep=';')

print("SUCESSO! População adicionada à base de dados.")
print(f"Total de linhas processadas: {len(df_final_enriquecido)}")
display(df_final_enriquecido[['Nome_Municipio', 'Tipo_Dado', 'Internacoes', 'Populacao', 'Taxa_Internacao_100k']].head())

SUCESSO! População adicionada à base de dados.
Total de linhas processadas: 13630


,Nome_Municipio,Tipo_Dado,Internacoes,Populacao,Taxa_Internacao_100k
0,ADAMANTINA,Realizado,397,35673.0,1112.886497
1,AGUAI,Realizado,7,32886.0,21.285653
2,AGUAS DE LINDOIA,Realizado,107,18257.0,586.076573
3,AGUDOS,Realizado,73,38988.0,187.237099
4,ALTINOPOLIS,Realizado,93,17197.0,540.791999


In [12]:
import pandas as pd
import os

caminho_dados = r'C:\Users\biama\OneDrive\Documentos\HealthOps-AI\data\raw'

# 1. Carregar a Base Analítica original
caminho_base = os.path.join(caminho_dados, 'healthops_base_analitica_mvp.csv')
df_analitica = pd.read_csv(caminho_base, sep=';')

# 2. Carregar e limpar os dados de População do IBGE (pulando a primeira linha de cabeçalho)
caminho_ibge = os.path.join(caminho_dados, 'ibge_populacao_sp.csv')
df_ibge = pd.read_csv(caminho_ibge, skiprows=1, encoding='utf-8', on_bad_lines='skip')

# Criar um DataFrame auxiliar só com Código IBGE e População
df_pop_limpa = pd.DataFrame()
df_pop_limpa['Cod_IBGE'] = df_ibge.iloc[:, 1].astype(str).str[:6] # Corta para 6 dígitos
df_pop_limpa['Populacao'] = pd.to_numeric(df_ibge.iloc[:, 7], errors='coerce').fillna(0)

# 3. Cruzar (Merge) a População com a Base Analítica
df_analitica['Cod_IBGE'] = df_analitica['Cod_IBGE'].astype(str).str.strip()
df_analitica_enriquecida = pd.merge(
    df_analitica,
    df_pop_limpa,
    on='Cod_IBGE',
    how='left'
)

# 4. Calcular a Taxa de Internação por 100k habitantes
def taxa_por_100k(row):
    if row['Populacao'] > 0:
        return (row['Internacoes'] / row['Populacao']) * 100000
    return 0

df_analitica_enriquecida['Taxa_Internacao_100k'] = df_analitica_enriquecida.apply(taxa_por_100k, axis=1)

# 5. Salvar e substituir a Base Analítica original com os novos dados
df_analitica_enriquecida.to_csv(caminho_base, index=False, encoding='utf-8-sig', sep=';')

print("SUCESSO! População e Taxa por 100k adicionadas à Base Analítica (Histórica).")
print(f"Total de linhas processadas: {len(df_analitica_enriquecida)}")
display(df_analitica_enriquecida[['Nome_Municipio', 'Internacoes', 'Populacao', 'Taxa_Internacao_100k']].head())

SUCESSO! População e Taxa por 100k adicionadas à Base Analítica (Histórica).
Total de linhas processadas: 11656


,Nome_Municipio,Internacoes,Populacao,Taxa_Internacao_100k
0,ADAMANTINA,397,35673.0,1112.886497
1,AGUAI,7,32886.0,21.285653
2,AGUAS DE LINDOIA,107,18257.0,586.076573
3,AGUDOS,73,38988.0,187.237099
4,ALTINOPOLIS,93,17197.0,540.791999
